# PGJANET Save and Reload Demo
This notebook shows how to save a `PGJANET_NeuralNetwork` to disk and reload it directly using the constructor.

In [1]:
from sparseDPD import DataManager, PGJANET_NeuralNetwork

# Same dataset/training style as refactored_notebook.ipynb
simpleDataManager = DataManager(
    filepath='UCD_datasets/PA_IO.mat',
    num_training_points=19000,
    num_validaiton_points=1000,
    num_test_points=2000,
)

model_path = 'pgjanet_checkpoint.pt'

PGJANET_forward_nn = PGJANET_NeuralNetwork(
    num_memory_levels=50,
    model_type='PGJANETNetwork',
    forward_model=True,
    seq_stride=10,
    batch_size=32,
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PGJANET_forward_nn.get_best_model(
    num_epochs=2,
    training_dataset=simpleDataManager.training_dataset,
    validation_dataset=simpleDataManager.validation_dataset,
    learning_rate=1e-2,
)

fwd_nmse = PGJANET_forward_nn.calculate_forward_nmse(simpleDataManager.test_dataset)
print(f"Trained PGJANET NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
PGJANET_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")

Using cpu device

Best model from epoch 2 with validation loss: 4.6379e-02
Trained PGJANET NMSE: -23.35 dB
Best epoch: 2
Saved trained model to: pgjanet_checkpoint.pt


In [2]:
import torch

loaded_pgjanet = PGJANET_NeuralNetwork(
    num_memory_levels=1,   # placeholders; overwritten by checkpoint
    hidden_size=4,
    seq_len=1,
    nn_file_path=model_path,
)

# Verify identical learned parameters.
same_params = all(
    torch.equal(v, loaded_pgjanet.nn_model.state_dict()[k])
    for k, v in PGJANET_forward_nn.nn_model.state_dict().items()
)

# Use the class NMSE method directly before and after serialization.
nmse_before = PGJANET_forward_nn.calculate_forward_nmse(simpleDataManager.test_dataset)
nmse_after = loaded_pgjanet.calculate_forward_nmse(simpleDataManager.test_dataset)
nmse_abs_diff = abs(nmse_before - nmse_after)

print("Parameters match:", same_params)
print(f"NMSE before save: {nmse_before:.12f} dB")
print(f"NMSE after load : {nmse_after:.12f} dB")
print(f"|NMSE diff|     : {nmse_abs_diff:.3e} dB")
print("NMSE unchanged (within 1e-10):", nmse_abs_diff < 1e-10)

Using cpu device
Parameters match: True
NMSE before save: -23.346904741537 dB
NMSE after load : -23.346904741537 dB
|NMSE diff|     : 0.000e+00 dB
NMSE unchanged (within 1e-10): True


In [3]:
# Now train a PNTDNN and see if you can reload it with the same NMSE

from sparseDPD import PNTDNN_NeuralNetwork


PNTDNN_forward_nn = PNTDNN_NeuralNetwork(
    num_memory_levels=50,
    model_type='OneLayerNetwork',
    forward_model=True)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PNTDNN_forward_nn.get_best_model(
    num_epochs=2,
    training_dataset=simpleDataManager.training_dataset,
    validation_dataset=simpleDataManager.validation_dataset,
    learning_rate=1e-2,
)

fwd_nmse = PNTDNN_forward_nn.calculate_forward_nmse(simpleDataManager.test_dataset)
print(f"Trained PNTDNN NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
model_path_pntdnn = 'pntdnn_checkpoint.pt'
PNTDNN_forward_nn.write_nn_to_file(model_path_pntdnn)
print(f"Saved trained PNTDNN model to: {model_path_pntdnn}")    


Using cpu device

Best model from epoch 2 with validation loss: 9.8535e-01
Trained PNTDNN NMSE: -19.35 dB
Best epoch: 2
Saved trained PNTDNN model to: pntdnn_checkpoint.pt


In [5]:
# Now reload the PNTDNN and check NMSE again
loaded_pntdnn = PNTDNN_NeuralNetwork(
    num_memory_levels=1,   # placeholders; overwritten by checkpoint
    nn_file_path=model_path_pntdnn,
)

nmse_before = PNTDNN_forward_nn.calculate_forward_nmse(simpleDataManager.test_dataset)
nmse_after = loaded_pntdnn.calculate_forward_nmse(simpleDataManager.test_dataset)
nmse_abs_diff = abs(nmse_before - nmse_after)

print("Parameters match:", same_params)
print(f"NMSE before save: {nmse_before:.12f} dB")
print(f"NMSE after load : {nmse_after:.12f} dB")
print(f"|NMSE diff|     : {nmse_abs_diff:.3e} dB")
print("NMSE unchanged (within 1e-10):", nmse_abs_diff < 1e-10)


Using cpu device
Parameters match: True
NMSE before save: -19.348365451182 dB
NMSE after load : -19.348365451182 dB
|NMSE diff|     : 0.000e+00 dB
NMSE unchanged (within 1e-10): True
